# Acquired vs pre-existing resistance — the resistance *origin* as a model-selection axis

**NOT FOR CLINICAL USE.** Population/trial-level only; illustrative, unverified parameters. Executed in CI (nbmake).

v0.24 made the resistance *mechanism* a model-selection axis (phenomenological Claret λ vs a mechanistic resistant subclone). That mechanistic model encodes one *origin*: a **pre-existing** resistant clone (`R0 > 0`). The other canonical origin is **acquired** resistance — no resistant cell at baseline, but under drug pressure sensitive cells convert to resistant at switching rate `α` (resistance is *generated by the treatment*):

```
dS/dt = (kg − kd·E)·S − α·E·S      sensitive: grow, killed, AND switch out
dR/dt =  kgr·R        + α·E·S      resistant: grow + the acquired influx
R(0)  = R0 = 0
```

With `kg`/`kd`/`kgr` matched to the pre-existing two-population model, the *only* difference is the resistance **origin**. This is `docs/specs/research/acquired-resistance.md`.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import onkos
from onkos.response import time_to_progression

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}
t = np.linspace(0, 156, 313)
ACQ = 'resistance.nsclc_first_line.acquired'
PRE = 'resistance.nsclc_first_line.two_population'
print(f'{len(ds)} records | onkos {onkos.__version__}')

## Same week-8, different tail

Matched on kg/kd/kgr, the two origins agree at week 8 and on the week-8-driven OS surrogate — but the acquired mechanism has a shallower nadir (the drug that kills also *generates* resistance) and progresses earlier.

In [ ]:
acq = onkos.simulate(ds, ACQ, context=ctx, drug_effect=1.0, t=t)
pre = onkos.simulate(ds, PRE, context=ctx, drug_effect=1.0, t=t)
print(f"{'origin':<14}{'wk8 (mm)':>10}{'nadir (mm)':>12}{'RECIST TTP':>12}{'median OS':>11}")
for name, tr in [('acquired', acq), ('pre-existing', pre)]:
    print(f"{name:<14}{tr.tumor_size[16]:>10.0f}{tr.tumor_size.min():>12.1f}"
          f"{time_to_progression(t, tr.tumor_size):>11.0f}w{tr.median_os:>11.0f}")
print('\n-> same week-8 OS (the surrogate is blind to the origin); different nadir depth + progression time (the tail sees it)')

## Strict generalization: α=0 recovers the pre-existing model

The acquired kernel is a superset — set `α=0` and it equals the pre-existing two-population kernel exactly (same kinetics).

In [ ]:
from onkos.export.reference import KERNELS
acq_rhs, two_rhs = KERNELS['acquired_resistance'].rhs, KERNELS['two_population_resistance'].rhs
v = {'kg': 0.021, 'kd': 0.3, 'kgr': 0.025, 'E': 1.0}
y = [40.0, 5.0]
print('acquired(α=0) rhs:', acq_rhs(0.0, y, {**v, 'alpha': 0.0}))
print('two-population rhs:', two_rhs(0.0, y, v))
print('switching flux conserves total dV/dt (α terms cancel):',
      np.isclose(sum(acq_rhs(0.0, y, {**v, 'alpha': 0.0})), sum(acq_rhs(0.0, y, {**v, 'alpha': 0.05}))))

## The acquired model is an ordinary TGI record

It joins the NSCLC compare set, exports through every format (round-trip validated), and is read by the existing PFS-route machinery — no new module needed. Guardrails ride through: out-of-context transport floors the tier to D.

In [ ]:
ids = [r.record_id for r in onkos.compare(ds, purpose='tgi', context=ctx).included]
print('in NSCLC compare set:', ACQ in ids, f'({len(ids)} models)')
print('tier in NSCLC      :', onkos.simulate(ds, ACQ, context=ctx, drug_effect=1.0).tier)
print('tier transported   :', onkos.simulate(ds, ACQ, context={'tumor_type':'melanoma','line':'first'}, drug_effect=1.0).tier, '(floored to D)')

## The takeaway

A resistance model carries a silent **origin** assumption — pre-existing or acquired. Matched on every shared parameter, the origin is invisible to the week-8 OS surrogate (OS ~92 vs 94 wk) yet drives a shallower response and earlier progression. Onkos makes the origin two explicit, comparable models and the divergence a measured quantity — never picking a side, never recommending a therapy or schedule. CI enforces it (`tests/test_acquired.py`).